# 02 — Embedding Exploration

**Goal**: Understand how Takens embedding parameters (delay τ, dimension d)
affect the topology of embedded light curves, and calibrate optimal parameters
for different source types and SNR levels.

Topics:
1. Optimal delay via mutual information
2. Optimal dimension via false nearest neighbors
3. Effect of interpolation strategy
4. Multi-band embedding approaches

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from void.data.synthetic import generate_periodic, generate_stochastic, generate_noise
from void.embedding.takens import TakensEmbedder, optimal_delay, optimal_dimension
from void.topology.persistence import compute_persistence
from void.viz.plots import plot_embedding_2d, plot_persistence_diagram

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
rng = np.random.default_rng(42)

## 1. Optimal delay selection

The delay τ determines how far apart in time the coordinates of each
embedding vector are. Too small → points cluster near the diagonal.
Too large → coordinates become uncorrelated.

In [ ]:
lc_periodic = generate_periodic(snr=5.0, period=30.0, n_epochs=300, rng=rng)

tau_mi = optimal_delay(lc_periodic, max_lag=30, method="mutual_information")
tau_ac = optimal_delay(lc_periodic, max_lag=30, method="autocorrelation")
print(f"Optimal delay (mutual information): τ = {tau_mi}")
print(f"Optimal delay (autocorrelation):    τ = {tau_ac}")

# Visualize embeddings at different delays
delays = [1, tau_mi, tau_mi * 2, 15]
fig, axes = plt.subplots(1, len(delays), figsize=(4 * len(delays), 4))
for ax, delay in zip(axes, delays):
    embedder = TakensEmbedder(dimension=3, delay=delay)
    cloud = embedder.embed(lc_periodic)
    plot_embedding_2d(cloud, ax=ax, title=f"τ = {delay}")
fig.suptitle("Effect of Delay on Periodic Source Embedding", y=1.02)
fig.tight_layout()
plt.show()

## 2. Optimal dimension selection

The embedding dimension d must be large enough to unfold the attractor.
False nearest neighbors (FNN) measures when increasing d stops
resolving spurious self-intersections.

In [ ]:
d_opt = optimal_dimension(lc_periodic, delay=tau_mi, max_dim=8)
print(f"Optimal embedding dimension: d = {d_opt}")

# Compare persistence at different dimensions
dims = [2, 3, 4, 5]
fig, axes = plt.subplots(1, len(dims), figsize=(4 * len(dims), 4))
for ax, d in zip(axes, dims):
    embedder = TakensEmbedder(dimension=d, delay=tau_mi)
    cloud = embedder.embed(lc_periodic)
    pd = compute_persistence(cloud, maxdim=1)
    plot_persistence_diagram(pd, ax=ax, title=f"d = {d}")
    ax.text(0.05, 0.95, f"H₁ total: {pd.total_persistence(1):.2f}",
            transform=ax.transAxes, fontsize=8, va="top")
fig.suptitle("Persistence vs Embedding Dimension", y=1.02)
fig.tight_layout()
plt.show()

## 3. Interpolation strategy comparison

LSST cadence is irregular. We can either interpolate to a regular grid
or use the raw cadence directly. Let's see what difference it makes.

In [ ]:
strategies = ["linear", "cubic", "none"]
fig, axes = plt.subplots(2, len(strategies), figsize=(4 * len(strategies), 7))

for col, interp in enumerate(strategies):
    embedder = TakensEmbedder(dimension=3, delay=tau_mi, interpolation=interp)
    cloud = embedder.embed(lc_periodic)
    pd = compute_persistence(cloud, maxdim=1)

    plot_embedding_2d(cloud, ax=axes[0, col], title=f"Interp: {interp}")
    plot_persistence_diagram(pd, ax=axes[1, col], title=f"PD ({interp})")
    axes[1, col].text(0.05, 0.95, f"H₁ total: {pd.total_persistence(1):.2f}",
                      transform=axes[1, col].transAxes, fontsize=8, va="top")

fig.suptitle("Interpolation Strategy Comparison (Periodic, SNR=5)", y=1.02)
fig.tight_layout()
plt.show()

## 4. Low SNR regime

Repeat the parameter exploration at sub-threshold SNR to see how
the optimal parameters change when signal is faint.

In [ ]:
snr_levels = [1.0, 2.0, 3.0, 5.0, 10.0]
fig, axes = plt.subplots(2, len(snr_levels), figsize=(3.5 * len(snr_levels), 7))

for col, snr in enumerate(snr_levels):
    trial_rng = np.random.default_rng(rng.integers(0, 2**63))
    lc = generate_periodic(snr=snr, period=30.0, n_epochs=300, rng=trial_rng)
    embedder = TakensEmbedder(dimension=3, delay=2)
    cloud = embedder.embed(lc)
    pd = compute_persistence(cloud, maxdim=1)

    plot_embedding_2d(cloud, ax=axes[0, col], title=f"SNR={snr}")
    plot_persistence_diagram(pd, ax=axes[1, col], title=f"PD (SNR={snr})")
    axes[1, col].text(0.05, 0.95, f"H₁: {pd.total_persistence(1):.2f}",
                      transform=axes[1, col].transAxes, fontsize=8, va="top")

fig.suptitle("Embedding & Persistence Across SNR (Periodic)", y=1.02)
fig.tight_layout()
plt.show()

## Summary

Key takeaways for embedding parameter selection:

1. **Mutual information** gives a principled delay estimate that maximizes
   topological signal.
2. **Dimension 3** is generally sufficient for the light curve types we consider.
3. **Linear interpolation** provides a good balance between preserving signal
   structure and handling irregular cadence.
4. At **low SNR**, the embedding becomes increasingly noisy but the loop structure
   from periodic signals remains partially visible down to ~2σ.